# 03 — Machine breakdowns

Second ingredient of the twin: when LITHOGRAPHY machines break down, for how long,
and whether breakdowns matter enough to be simulated.

In [1]:
import numpy as np
import pandas as pd

from wafer_twin.data import load_tables, build_process

tables = load_tables()
process = build_process(tables)

## Breakdown time and repair time

Two quantities describe breakdowns:
- **time between failures** (mean = MTBF): how long a machine runs before breaking down
- **repair time** (mean = MTTR): how long a breakdown lasts

First we check how much capacity breakdowns actually remove, before deciding how to model them.

In [2]:
# prepare the breakdowns' table
bd = tables["machine_breakdown_table"].merge(
    tables["machine_table"][["MACHINE", "WORKSHOP"]], on="MACHINE", how="left"
)
bd["TTR_MIN"] = (bd["BREAKDOWN_END_DATE"] - bd["BREAKDOWN_BEGIN_DATE"]).dt.total_seconds() / 60

# time running since the previous breakdown of the same machine ended
bd = bd.sort_values(["MACHINE", "BREAKDOWN_BEGIN_DATE"]).reset_index(drop=True)
bd["TBF_MIN"] = (
    bd["BREAKDOWN_BEGIN_DATE"] - bd.groupby("MACHINE")["BREAKDOWN_END_DATE"].shift(1)
).dt.total_seconds() / 60

print("missing workshop:", bd["WORKSHOP"].isna().sum())
print(f"{len(bd):,} breakdowns")
bd.groupby("WORKSHOP")["TTR_MIN"].describe().round(2)

missing workshop: 0
277,107 breakdowns


,count,mean,std,min,25%,50%,75%,max
WORKSHOP,,,,,,,,
ETCHING,149830.0,2.60,4.42,0.0,0.47,1.22,2.81,131.37
IMPLEMENTATION,49566.0,0.40,0.48,0.0,0.09,0.23,0.51,6.31
LITHOGRAPHY,77711.0,0.38,0.79,0.0,0.05,0.14,0.36,16.41


In [3]:
# lost capacity
horizon_min = (
    process["PROCESS_END_DATE"].max() - process["PROCESS_BEGIN_DATE"].min()
).total_seconds() / 60

n_machines = tables["machine_table"]["WORKSHOP"].value_counts()
downtime = bd.groupby("WORKSHOP")["TTR_MIN"].sum()

# share of total machine time spent broken down
(downtime / (n_machines * horizon_min)).round(5)

WORKSHOP
ETCHING           0.00618
IMPLEMENTATION    0.00062
LITHOGRAPHY       0.00071
dtype: float64

In [4]:
# MTBF and MTTR of lithography machines
per_machine = (
    bd[bd["WORKSHOP"] == "LITHOGRAPHY"]
    .groupby("MACHINE")
    .agg(n=("TTR_MIN", "size"), mtbf_min=("TBF_MIN", "mean"), mttr_min=("TTR_MIN", "mean"))
    .join(tables["machine_table"].set_index("MACHINE")[["lambda_breakdown_mean", "mu_breakdown_mean"]])
)

# if lambda and mu are rates per minute, both products should be close to 1
per_machine["mtbf_x_lambda"] = per_machine["mtbf_min"] * per_machine["lambda_breakdown_mean"]
per_machine["mttr_x_mu"] = per_machine["mttr_min"] * per_machine["mu_breakdown_mean"]

print(per_machine[["mtbf_x_lambda", "mttr_x_mu"]].describe().round(3))
per_machine.round(3).head(10)

       mtbf_x_lambda  mttr_x_mu
count         40.000     40.000
mean           1.117      1.009
std            0.100      0.034
min            0.963      0.964
25%            1.059      0.986
50%            1.101      1.007
75%            1.177      1.020
max            1.400      1.144


,n,mtbf_min,mttr_min,lambda_breakdown_mean,mu_breakdown_mean,mtbf_x_lambda,mttr_x_mu
MACHINE,,,,,,,
MACHINE_0001,1044,1008.645,0.297,0.001,3.507,1.111,1.041
MACHINE_0002,2165,485.591,0.255,0.002,4.006,1.176,1.022
MACHINE_0003,228,4593.219,0.432,0.000,2.307,0.963,0.997
MACHINE_0004,1976,531.992,0.101,0.002,9.754,1.118,0.983
MACHINE_0005,4295,245.038,0.070,0.005,14.241,1.187,0.996
MACHINE_0006,503,2069.608,0.232,0.001,4.241,1.055,0.984
MACHINE_0007,89,11591.897,0.708,0.000,1.475,1.024,1.044
MACHINE_0008,2911,361.184,0.424,0.003,2.374,1.177,1.006
MACHINE_0009,1363,771.408,0.388,0.001,2.623,1.113,1.018


### Takeaways

- Breakdowns are micro-stops: in LITHOGRAPHY, mean 0.38 min (~23 s), median ~9 s, max 16 min
- Capacity lost to breakdowns: 0.071% in LITHOGRAPHY, 0.062% in IMPLEMENTATION, 0.62% in ETCHING
- MTBF varies a lot between machines: from ~4 h (MACHINE_0005) to ~8 days (MACHINE_0007)
- `mu_breakdown_mean` is a repair rate per minute: MTTR × μ = 1.01 on average (0.96–1.14)
- `lambda_breakdown_mean` is a breakdown rate per minute: MTBF × λ = 1.12 on average (0.96–1.40),
  so observed MTBF is ~12% longer than 1/λ (cause unclear)
- Decision: breakdowns are left out of the first version of the twin
  (negligible capacity loss, and short stops may already be included in measured processing times).
  λ and μ are kept to add them later as an option